# TP : Diagnostic de la Rouille Polysora sur les Feuilles de Maïs à Madagascar

Classification d'images de feuilles de maïs : **Saine (0)** vs **Malade (1)** (Rouille Polysora, *Puccinia polysora*).

Ce notebook regroupe les 4 parties du TP :
1. **Feature Engineering** — du pixel aux caractéristiques
2. **Indice Max-Minority** — métrique de pureté personnalisée
3. **Arbres & Forêts** — from scratch vs scikit-learn
4. **Application Streamlit** — voir `app.py`

## Partie 1 : Feature Engineering — Du Pixel aux Caractéristiques

On extrait pour chaque image trois descripteurs :
- **X1 `pct_rouille`** : pourcentage de pixels de teinte rouille (masque HSV)
- **X2 `rugosite`** : variance des gradients de Sobel (texture)
- **X3 `ratio_vert`** (feature personnelle) : pourcentage de pixels verts.
  *Justification agronomique* : une feuille saine est riche en chlorophylle (verte) ;
  la rouille détruit les cellules foliaires et réduit la surface verte.

In [1]:
import numpy as np
import pandas as pd
from feature_engineering import construire_dataframe

df = construire_dataframe('dataset')
df.to_csv('features.csv', index=False)
print('Nombre d\'images :', len(df))
df.head()

Nombre d'images : 400


,ID_Image,pct_rouille,rugosite,ratio_vert,label_malade
0,saine_000.jpg,0.000000,1209.581660,0.984665,0
1,saine_001.jpg,0.000000,744.938414,0.999985,0
2,saine_002.jpg,0.000000,3010.486529,0.781952,0
3,saine_003.jpg,0.000000,906.215302,0.944031,0
4,saine_004.jpg,0.000031,555.715848,0.878983,0


In [2]:
# Moyenne des features par classe
df.groupby('label_malade')[['pct_rouille', 'rugosite', 'ratio_vert']].mean()

,pct_rouille,rugosite,ratio_vert
label_malade,,,
0,0.000781,1845.532665,0.852773
1,0.161601,7256.573448,0.411054


## Partie 2 : L'Indice "Max-Minority"

Pureté d'un nœud : $P(t) = \max_{c \in \{0,1\}} \frac{n_c}{N}$

Pureté pondérée d'un split : $P_{split} = \frac{|G|}{N} P(G) + \frac{|D|}{N} P(D)$ — on cherche le seuil qui **maximise** $P_{split}$.

In [3]:
from max_minority import purete, trouver_meilleur_split

# Test rapide sur la variable pct_rouille
seuil, p = trouver_meilleur_split(df['pct_rouille'].values, df['label_malade'].values)
print(f'Meilleur seuil pct_rouille = {seuil:.5f}  |  Pureté = {p:.4f}')

seuil_v, p_v = trouver_meilleur_split(df['ratio_vert'].values, df['label_malade'].values)
print(f'Meilleur seuil ratio_vert = {seuil_v:.5f}  |  Pureté = {p_v:.4f}')

Meilleur seuil pct_rouille = 0.00661  |  Pureté = 0.9825
Meilleur seuil ratio_vert = 0.69079  |  Pureté = 0.9300


## Partie 3 : Arbres et Forêts — From Scratch vs Scikit-Learn

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from models import build_tree, predire_arbre_batch, RandomForestMaxMinority

feature_cols = ['pct_rouille', 'rugosite', 'ratio_vert']
X = df[feature_cols].values
y = df['label_malade'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train :', len(X_train), '| Test :', len(X_test))

Train : 320 | Test : 80


In [5]:
# 1) Arbre Max-Minority (from scratch)
arbre_mm = build_tree(X_train, y_train, depth=0, max_depth=5)
pred_1 = predire_arbre_batch(arbre_mm, X_test)

# 2) Random Forest Max-Minority (from scratch)
rf_mm = RandomForestMaxMinority(n_arbres=20, max_depth=5, random_state=42)
rf_mm.fit(X_train, y_train)
pred_2 = rf_mm.predict(X_test)

# 3) Arbre scikit-learn (Gini)
dt_sk = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
dt_sk.fit(X_train, y_train)
pred_3 = dt_sk.predict(X_test)

# 4) Random Forest scikit-learn (Gini, n=100)
rf_sk = RandomForestClassifier(n_estimators=100, criterion='gini', random_state=42)
rf_sk.fit(X_train, y_train)
pred_4 = rf_sk.predict(X_test)

resultats = {
    'Arbre Max-Minority': pred_1,
    'RF Max-Minority': pred_2,
    'Arbre Gini (sklearn)': pred_3,
    'RF Gini (sklearn)': pred_4,
}

In [6]:
# Tableau comparatif : Accuracy / Précision / Rappel
rows = []
for nom, pred in resultats.items():
    rows.append({
        'Modèle': nom,
        'Accuracy': accuracy_score(y_test, pred),
        'Précision': precision_score(y_test, pred, zero_division=0),
        'Rappel': recall_score(y_test, pred, zero_division=0),
    })
pd.DataFrame(rows).set_index('Modèle').round(4)

,Accuracy,Précision,Rappel
Modèle,,,
Arbre Max-Minority,1.0,1.0,1.0
RF Max-Minority,1.0,1.0,1.0
Arbre Gini (sklearn),1.0,1.0,1.0
RF Gini (sklearn),1.0,1.0,1.0


In [7]:
# Matrices de confusion
for nom, pred in resultats.items():
    cm = confusion_matrix(y_test, pred)
    print(f'--- {nom} ---')
    print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
    print(f'  FN={cm[1,0]}  TP={cm[1,1]}\n')

--- Arbre Max-Minority ---
  TN=40  FP=0
  FN=0  TP=40

--- RF Max-Minority ---
  TN=40  FP=0
  FN=0  TP=40

--- Arbre Gini (sklearn) ---
  TN=40  FP=0
  FN=0  TP=40

--- RF Gini (sklearn) ---
  TN=40  FP=0
  FN=0  TP=40



In [8]:
# Importance des variables (Random Forest sklearn)
pd.Series(rf_sk.feature_importances_, index=feature_cols).sort_values(ascending=False)

pct_rouille    0.398850
rugosite       0.366208
ratio_vert     0.234942
dtype: float64

## Partie 3 — Analyse Critique

**1. Comportement de l'algorithme personnel vs scikit-learn.**
L'arbre et la forêt "from scratch" (Max-Minority) obtiennent des performances très proches de celles de scikit-learn (Gini). C'est attendu : sur ce jeu de features, les classes sont fortement séparables (le `pct_rouille` et le `ratio_vert` discriminent nettement les deux classes), donc les deux métriques de pureté trouvent des seuils équivalents.

**2. Pourquoi la Forêt Aléatoire améliore la robustesse.**
Un arbre unique a une **forte variance** : il peut sur-apprendre des particularités du jeu d'entraînement. La Forêt Aléatoire entraîne plusieurs arbres sur des sous-échantillons différents (**bagging**, tirage avec remplacement) puis agrège par **vote majoritaire**. Cette agrégation réduit la variance et atténue les erreurs individuelles : le modèle généralise mieux sur des images de terrain bruitées.

**3. Discussion agronomique (Madagascar) — quel modèle déployer ?**
- Un **Faux Négatif** (feuille malade classée saine) est le plus dangereux : l'épidémie se propage et peut détruire les récoltes. Il faut donc **maximiser le Rappel** (sensibilité) sur la classe "malade".
- Un **Faux Positif** (feuille saine classée malade) coûte un traitement fongicide inutile — gênant mais bien moins grave.

**Recommandation : déployer le `RandomForestClassifier` (scikit-learn, Gini, n=100)**. Il combine un rappel élevé (peu de feuilles malades manquées) et la robustesse de l'agrégation, ce qui est crucial pour un usage terrain par des techniciens agricoles.

In [9]:
# Sauvegarde du modèle pour l'application Streamlit
import pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(rf_sk, f)
with open('model_meta.pkl', 'wb') as f:
    pickle.dump({'feature_cols': feature_cols}, f)
print('Modèle sauvegardé → model.pkl')

Modèle sauvegardé → model.pkl


## Partie 4 : Application Web Streamlit

L'application est dans `app.py`. Lancer :

```bash
streamlit run app.py
```

Elle propose :
- **Upload + prédiction temps réel** : on téléverse une image, l'app extrait `pct_rouille`/`rugosite`/`ratio_vert` et affiche une alerte rouge (Malade) ou un message vert (Saine).
- **Galerie d'historique** : les images analysées sont conservées dans `uploads/` et affichées en miniatures avec leur diagnostic.